# Review R01 — Regression diagnostics

**Short on time? Answer only the ★ questions (about 5 minutes each).** The others are optional practice; they come back in the April interview sprint.

One weekly session (~30-45 min). Each notebook's exam is appended under its own heading; once you have answered and Claude has reviewed, this review is closed and the next notebook starts a new one.

Almost no code: model outputs are printed as text, and you *read*, *predict*, or *explain*. Write 2-3 plain bullets per sub-part in **Your answer** (the way you'd say it in an interview). Leave **Review** alone; ask Claude to fill it in when you're done. Role tags: **DS**, **Causal**, **ML**; any question is fair game for any role.

## From notebook 11 — Regression diagnostics

### Q1 — A huge residual that doesn't matter
*Role lean: Causal*

META excess return regressed on `Mkt_RF`, 2,513 days: beta = 1.275 (SE 0.032). Residual SD = 0.0191; `Mkt_RF` SD = 0.0117; average leverage = 0.0008, and the 4/n cutoff for Cook's distance is 0.0016.

```text
date         META return   Mkt_RF   stud. resid   leverage   Cook's D   beta without that day
2022-10-27     -24.6%     -0.54%      -12.9       0.0005     0.039      1.271
2022-02-03     -26.4%     -2.48%      -12.6       0.0023     0.168      1.258
2020-03-16     -14.3%     -12.01%     +0.55       0.0425     0.007      1.279

Days with Cook's D above 4/n: 112 of 2513
```

(a) The first row is a −24.6% day with a studentized residual of −12.9, yet removing it barely moves beta. Why? (b) The last row has the highest leverage in the sample but a Cook's D of only 0.007. Why? (c) Interview version: "The 4/n rule flags 112 days. What do you do with that list, and would you delete the 2022-02-03 observation before reporting META's beta?"

**Your answer (Q1):**

* 

**Review (Q1):**

_pending_

### Q2 — Two tests, two different questions
*Role lean: ML*

XOM excess return regressed on `Mkt_RF`, 2,513 days: beta = 0.776.

```text
Breusch-Pagan (variance vs Mkt_RF):          p = 0.74
Ljung-Box, lag 10, residuals:                p = 0.001
Ljung-Box, lag 10, squared residuals:        p < 0.001
Lag-1 autocorrelation:   residuals 0.021   |   squared residuals 0.150
SE of beta:              classical 0.0255   |   HC3 0.0422

Residual SD by year:  2016 0.0099  2017 0.0068  2018 0.0103  2019 0.0089  2020 0.0243
                      2021 0.0168  2022 0.021  2023 0.0152  2024 0.013  2025 0.0138
```

(a) Breusch-Pagan does not reject, but Ljung-Box on the squared residuals rejects overwhelmingly. Are these in conflict? Say what each test actually asks. (b) HC3's SE is 65% larger than the classical SE even though Breusch-Pagan did not reject. Offer one reason, and say whether HC3 addresses what the Ljung-Box test found. (c) Interview version: "You need a one-day-ahead risk number for XOM. What does this output tell you about using a constant residual SD?"

**Your answer (Q2):**

* 

**Review (Q2):**

_pending_

### Q3 — Overlapping windows
*Role lean: Causal*

JPM excess return on `Mkt_RF` using rolling-sum returns over different windows (consecutive rows overlap). HAC uses `maxlags = window − 1`.

```text
window    n    beta    classical SE   HAC SE    Durbin-Watson   lag-1 residual corr
1 day   2513   1.053      0.0208        0.0377       2.05           -0.02
5 day   2509   1.074      0.0214        0.0428       0.44           +0.78
21 day  2493   1.080      0.0204        0.0808       0.13           +0.93
```

(a) Predict the 63-day row: will the Durbin-Watson be higher or lower than 0.13, and will the HAC SE be more or less than 4× the classical SE? (b) The classical SE stays near 0.02 at every window. Why does it fail to notice anything? (c) Interview version: "A teammate regresses 12-month overlapping returns, sampled monthly, and reports a t-statistic of 6. What is your concern, and name two ways to fix it?"

**Your answer (Q3):**

* 

**Review (Q3):**

_pending_

### Q4 — Robust standard errors are not a guarantee
*Role lean: DS*

AAPL excess return on `Mkt_RF`, 2016-2025 (2,513 days): beta = 1.166.

```text
SE of beta:  classical 0.0207  |  HC3 0.0276  |  HAC (maxlags=5) 0.0323
95% CI with HC3 SE: 1.111 to 1.220

By period (HC3 SEs):
2016-2020 (1,258 days)   beta 1.158   SE 0.0369   95% CI 1.086 to 1.230
2021-2025 (1,255 days)   beta 1.175   SE 0.0419   95% CI 1.092 to 1.257
```

A teammate says: "We used robust standard errors, so the interval 1.11-1.22 is trustworthy. I'll use beta = 1.17 to size next year's hedge." (a) What do robust standard errors fix, and what is that interval still *not* a statement about? (b) Does the by-period table support a constant beta? How would you test that formally? (c) The universe of stocks in this project was chosen with hindsight, in 2026. How could that make AAPL's history unrepresentative?

**Your answer (Q4):**

* 

**Review (Q4):**

_pending_

### Q5 — Is the kink real?
*Role lean: ML · DS · interview*

One regression per stock for all 66 stocks (SPY excluded): `y ~ Mkt_RF + SMB + HML + I(np.minimum(Mkt_RF, 0))`. The table counts stocks whose down-market (hinge) term has |t| > 2.

```text
                 full sample              Feb 15 - Apr 30 2020 removed
classical SEs    11  (4 positive, 7 negative)     1  (1 positive, 0 negative)
HC3 SEs           1  (1 positive, 0 negative)     1  (1 positive, 0 negative)

With HC3, the one flagged stock is SPG on the full sample and PFE with Covid removed.
No stock is flagged in both samples.
```

(a) What does the drop from 11 to 1 (switching to HC3, or removing Covid) say about the 11 flags? (b) How many of the 66 would you expect to clear |t| > 2 by pure chance if no stock had a kinked beta, and how does that change what you make of the single remaining flag? (c) Interview version: "Would you add a nonlinear down-market term to the firm's beta model? What evidence would you want first?"

**Your answer (Q5):**

* 

**Review (Q5):**

_pending_

## From notebook 13 — Naive Bayes and LDA

### ★ Q6 — Naive Bayes by hand
*Role lean: DS · interview*

Training data: 4,930 customers, 26.5% churned. Share of each group with the trait:

```text
                               among churners   among stayers
Contract = Month-to-month           0.886            0.427
TechSupport = No                    0.773            0.387
OnlineSecurity = No                 0.784            0.390

Actual churn rate among the 1,501 training customers with all three traits: 55.0%
```

(a) Compute the naive-Bayes P(churn) for a customer with all three traits.
(b) It lands well above 55%. Explain why in one sentence that is about these particular three traits.
(c) An interviewer asks: "So Naive Bayes is useless here?" What do you say?

**Your answer (Q6):**

* 

**Review (Q6):**

_pending_

### Q7 — Feeding Naive Bayes the same clue twice
*Role lean: ML*

`CategoricalNB` on four features (Contract, InternetService, PaymentMethod, tenure_band), then the same model with extra **exact copies** of the Contract column added. Test set: 2,113 customers, 26.6% churned; cutoff 0.5.

```text
                          accuracy   churners caught   customers flagged   share with P >= 0.9   actual churn among them
4 features                  0.774         60.6%               596                 0.0%                    —
+1 copy of Contract         0.744         79.5%               873                 8.5%                  70.6%
+2 copies of Contract       0.695         84.8%              1035                21.7%                  58.6%
```

(a) No new information was added. Why did the predicted probabilities get more extreme?
(b) Accuracy fell while churners caught rose. Explain both with one mechanism.
(c) Name a situation in real data where this "copy" happens without anyone duplicating a column on purpose.

**Your answer (Q7):**

* 

**Review (Q7):**

_pending_

### ★ Q8 — Which feature "matters most"?
*Role lean: Causal · DS · interview*

LDA on standardized inputs (fit on train, scored on the 2,113-customer test set):

```text
                                     tenure   MonthlyCharges   TotalCharges   test accuracy
Model A: tenure + Monthly + Total    -0.415       +1.447          -1.156          0.779
Model B: tenure + Monthly            -1.210       +0.883            —             0.782

corr(tenure, TotalCharges) = 0.83   |   R² of TotalCharges on tenure + MonthlyCharges = 0.90 (VIF ≈ 9.7)
```

A teammate reads Model A and says: "Monthly charges drive churn about 3.5 times more than tenure, so cutting prices is our best lever."

(a) What happened to the weights between Model A and Model B, and why did accuracy barely move?
(b) Name two separate problems with the teammate's conclusion.
(c) What would you need before claiming that price cuts reduce churn?

**Your answer (Q8):**

* 

**Review (Q8):**

_pending_